# Classify Job Postings

Run `3-analyse-aie-job-postings.ipynb` first so that `jobs/1-scraped_jobs.jsonl` exists.

This notebook reads the scraped jobs, uses an LLM to classify whether each role is truly an AI engineering role, and writes the accepted jobs to `jobs/2-llm_filtered_jobs.jsonl`.


In [1]:
import json
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import HTML, display

screening_jsonl_path = Path("jobs") / "1-scraped_jobs.jsonl"
llm_filtered_jsonl_path = Path("jobs") / "2-llm_filtered_jobs.jsonl"

if not screening_jsonl_path.exists():
    raise FileNotFoundError(
        f"Scraped jobs JSONL file not found: {screening_jsonl_path.resolve()}. Run 3-analyse-aie-job-postings.ipynb first."
    )

screening_source_jobs = pd.read_json(screening_jsonl_path, lines=True)

if screening_source_jobs.empty:
    raise ValueError("The scraped jobs JSONL file is empty. Run 3-analyse-aie-job-postings.ipynb first.")

if "description" not in screening_source_jobs.columns:
    raise KeyError("The scraped jobs JSONL file must contain a 'description' column.")

screening_source_jobs = screening_source_jobs[screening_source_jobs["description"].notna()].copy()
screening_source_jobs["description"] = screening_source_jobs["description"].astype(str).str.strip()
screening_source_jobs = screening_source_jobs[screening_source_jobs["description"] != ""].copy()
screening_source_jobs = screening_source_jobs.reset_index(drop=True)

if screening_source_jobs.empty:
    raise ValueError("The scraped jobs JSONL file contains no usable job descriptions for LLM screening.")

client = OpenAI()

screening_instructions = """
You classify whether a job posting is truly for an AI Engineering role.

AI Engineering definition:
- AI engineering means building applications on top of foundation models.
- Traditional ML engineering focuses on building, training, or tuning models; AI engineering primarily leverages existing models.
- An AI Engineer is not mainly a model researcher and does not primarily build models from scratch.
- AI engineering is closer to product and application engineering than to core ML research.

Decision rules:
- Set is_ai_engineering_role to true when the main responsibility is building product or application features on top of foundation models or LLMs.
- Set is_ai_engineering_role to false when the role is mainly traditional software engineering, data science, analytics, ML research, model training, classical ML engineering, MLOps or platform work, or something else where AI application work is not the core responsibility.
- If the posting is ambiguous or unclear, set is_ai_engineering_role to false.
""".strip()

screening_schema = {
    "type": "object",
    "properties": {
        "is_ai_engineering_role": {"type": "boolean"},
        "reason": {"type": "string"},
    },
    "required": ["is_ai_engineering_role", "reason"],
    "additionalProperties": False,
}

max_description_chars = 10000
screening_results = []

for i, (_, job) in enumerate(screening_source_jobs.iterrows(), start=1):
    title = "" if pd.isna(job.get("title")) else str(job.get("title")).strip()
    company = "" if pd.isna(job.get("company")) else str(job.get("company")).strip()
    job_url = "" if pd.isna(job.get("job_url")) else str(job.get("job_url")).strip()
    description = str(job.get("description", "")).strip()

    description_for_prompt = description[:max_description_chars]
    if len(description) > max_description_chars:
        description_for_prompt += "\n\n[Description truncated for screening.]"

    print(f"Screening job {i}/{len(screening_source_jobs)}: {title} @ {company}")

    response = client.responses.create(
        model="gpt-5.4-nano",
        instructions=screening_instructions,
        input=f"""Classify this job posting.\n\nTitle: {title}\nCompany: {company}\nURL: {job_url}\n\nDescription:\n{description_for_prompt}""",
        max_output_tokens=120,
        text={
            "format": {
                "type": "json_schema",
                "name": "ai_engineering_job_screening",
                "schema": screening_schema,
                "strict": True,
            },
            "verbosity": "low",
        },
    )

    result = json.loads(response.output_text)
    is_ai_engineering_role = bool(result.get("is_ai_engineering_role", False))
    reason = str(result.get("reason", "")).strip()

    screening_results.append(
        {
            "is_ai_engineering_role": is_ai_engineering_role,
            "llm_reason": reason,
        }
    )

screening_results_df = pd.DataFrame(screening_results)
screened_jobs = pd.concat([screening_source_jobs, screening_results_df], axis=1)
screened_jobs["is_ai_engineering_role"] = screened_jobs["is_ai_engineering_role"].fillna(False)
screened_jobs["llm_reason"] = screened_jobs["llm_reason"].fillna("")

ai_engineer_jobs = screened_jobs[screened_jobs["is_ai_engineering_role"]].copy()
non_ai_engineer_jobs = screened_jobs[~screened_jobs["is_ai_engineering_role"]].copy()
ai_engineer_jobs.to_json(llm_filtered_jsonl_path, orient='records', lines=True, force_ascii=False)

print(f"Saved LLM-filtered jobs to: {llm_filtered_jsonl_path.resolve()}")
print(f"Jobs screened by LLM: {len(screened_jobs)}")
print(f"Jobs classified as AI engineering roles: {len(ai_engineer_jobs)}")
print(f"Jobs classified as not AI engineering or unclear: {len(non_ai_engineer_jobs)}")

screening_results_to_show = screened_jobs[["title", "company", "is_ai_engineering_role", "llm_reason", "job_url"]].copy()
screening_results_to_show["job_url"] = screening_results_to_show["job_url"].apply(
    lambda url: f'<a href=\"{url}\" target=\"_blank\">{url}</a>' if pd.notna(url) and str(url).strip() else ''
)

display(HTML(screening_results_to_show.to_html(escape=False, index=False)))


Screening job 1/60: AI Engineer (DevOps) @ Wipro
Screening job 2/60: Principal AI Engineer @ VideoAmp
Screening job 3/60: AI Engineer / Agentic and Generative AI @ Realign
Screening job 4/60: Software AI Engineer – Agentic AI-2 @ Realign
Screening job 5/60: AI Engineer, Enterprise AI (Principal/Sr. Principal/Distinguished) @ Palo Alto Networks
Screening job 6/60: Junior Applied AI Engineer- HRIS @ Fortinet
Screening job 7/60: Microservices Tech Lead – AI Engineer/Analyst - Vice President – TAMPA @ Information Technology Senior Management Forum
Screening job 8/60: Principal AI Engineer @ Burrell Behavioral Health
Screening job 9/60: Principal, Software Engineer - GenAi @ Walmart
Screening job 10/60: AI Engineer @ American Technology Consulting LLC
Screening job 11/60: AI/ML Engineer [REMOTE JOB] @ BAE Systems USA
Screening job 12/60: Gen AI Engineer @ Persistent Systems
Screening job 13/60: Senior Gen AI Engineer (USC/GC) W2 Role @ Hudson Manpower
Screening job 14/60: Sr. AI-ML Engineer

title,company,is_ai_engineering_role,llm_reason,job_url
AI Engineer (DevOps),Wipro,False,"The role is titled “AI Engineer (DevOps)” but the responsibilities emphasize DevOps/enterprise platform tooling (AWS/Kubernetes/Docker, L1–L2 support, Slack/user issue handling) rather than building product/application features on top of LLM foundation models. While it mentions LLM concepts (RAG, inference, drift), the core focus appears to be operational support and platform/DevOps work.",https://www.indeed.com/viewjob?jk=f827c33d4ee013f2
Principal AI Engineer,VideoAmp,True,"Role is focused on building and operating agentic workflow systems, MCP/tool layers, evaluation pipelines, and LLM abstraction layers—i.e., production AI infrastructure and developer-facing platform features on top of foundation model APIs, not training new models from scratch.",https://www.indeed.com/viewjob?jk=a59140bb24e25483
AI Engineer / Agentic and Generative AI,Realign,True,"The role is focused on building and operating agentic/generative AI chat/LLM workflows (prompting, tool/RAG integration, evaluation, routing/guardrails) and delivering end-to-end applications across systems, rather than training foundational models from scratch.",https://www.indeed.com/viewjob?jk=dd8d94bee2e51705
Software AI Engineer – Agentic AI-2,Realign,False,"Although it mentions agentic LLM systems and orchestration frameworks, the description heavily emphasizes model-level work (fine-tuning/inference tooling with PyTorch/Hugging Face) and broad ML infrastructure/MLOps-style capabilities (inference, model gateways, evaluation/observability). It’s not clearly centered on building product features on top of existing foundation models versus also doing substantial traditional ML/model engineering, so it’s classified as not primarily AI Engineering.",https://www.indeed.com/viewjob?jk=8ac7b54fb8ca57c0
"AI Engineer, Enterprise AI (Principal/Sr. Principal/Distinguished)",Palo Alto Networks,True,"The role focuses on designing, building, and operationalizing AI-enabled enterprise workflows using LLM/agent patterns (RAG, tool-calling, orchestration, evaluation, observability) and integrating them into business systems—primarily application/product engineering on top of foundation models rather than training/researching models.",https://www.indeed.com/viewjob?jk=07564be4ba2a6d92
Junior Applied AI Engineer- HRIS,Fortinet,True,"The role focuses on building AI-powered HR applications (agents, chatbots, recommendations, predictive analytics, workflow automation) on top of LLMs/AI rather than doing core model research, with responsibilities centered on deploying user-facing AI features in an HRIS context.",https://www.indeed.com/viewjob?jk=19fc4813887bb16d
Microservices Tech Lead – AI Engineer/Analyst - Vice President – TAMPA,Information Technology Senior Management Forum,False,"The role is primarily a microservices/application development tech lead (Java/Spring Boot/Java Spark, architecture templates, coding/testing, leading development teams). It mentions AI tools broadly but does not focus on building AI/LLM-based applications on top of foundation models; it reads as traditional applications engineering/software leadership.",https://www.indeed.com/viewjob?jk=911b6e2b9ebbe41b
Principal AI Engineer,Burrell Behavioral Health,True,"Role focuses on building enterprise LLM-enabled analytics products (natural language-to-SQL, conversational query interfaces, prototype-to-production on Snowflake Cortex) and defining patterns for LLM-driven systems on the data platform, rather than researching or training foundation models from scratch.",https://www.indeed.com/viewjob?jk=09230ed99d1cba66
"Principal, Software Engineer - GenAi",Walmart,True,"The role focuses on integrating GenAI/LLMs into production e-commerce systems (React/Node/GraphQL, orchestration, RAG/agents, evaluation/monitoring) and building AI-powered customer features at scale, rather than training foundation models from scratch.",https://www.inde